# FM Agent — Workshop Tutorial

The agent built in this notebook is a **geospatial event analysis assistant** powered by Prithvi-EO-2.0. Given a natural-language request, it resolves the location (geocoding), checks Harmonized Landsat–Sentinel-2 imagery availability for the requested date range, and runs Prithvi inference for one of three supported tasks — **flood detection**, **burn-scar mapping**, or **crop-type classification** — then returns a narrative summary with output links plus a behind-the-scenes JSON audit log.

Everything that defines its behavior (scope, contexts, tool inventory, output format, guardrails, reasoning) lives as plain markdown files inside an **artifact** directory; the notebook wires that artifact to a model using `pydantic-ai` and `pydantic-ai-backends` (read-only `LocalBackend` + `ConsoleCapability`), so the agent reads its own prompt material from disk on demand via tool calls.

## 1. Imports and global config

In [ ]:
from pathlib import Path
from dataclasses import dataclass

from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic_ai import Agent
from pydantic_ai_backends import (
    LocalBackend,
    ConsoleCapability,
    READONLY_RULESET,
)

# Load API keys (e.g. OPENAI_API_KEY) from a local .env file into the process env.
load_dotenv()

# Global config — edit these to point at a different artifact or change models.
ARTIFACT_PATH = Path("artifacts/Prithvi_WOrkshop_Agent_artifact")
MODEL = "openai:gpt-5.2"
MODEL_SETTINGS = {"openai_reasoning_effort": "medium"}

## 2. Artifact structure

<details>
<summary>Details</summary>

An agent artifact follows this convention:

```
<workspace>/
  ├── resources/        ← user-uploaded reference material
  │   └── <files>
  ├── scope.md          ← agent purpose, users, workflow, success
  ├── contexts/         ← existing systems + context workspace
  │   ├── index.md      ← manifest written when contexts confirmed
  │   └── <topic>.md
  ├── tools/            ← tool inventory
  │   ├── index.md      ← manifest of tools
  │   └── <tool>/index.md
  ├── output.md         ← output format
  ├── reasoning.md      ← reasoning strategy
  ├── guardrails/       ← safety & guardrails
  │   ├── index.md
  │   └── <rule>.md
  └── agents.md         ← final assembled agent prompt (the entry point)
```

**`agents.md` is the entry point** — we load it verbatim as the system prompt. It internally references the other files by relative path (e.g. `contexts/hls_conventions.md`), and the agent pulls them in on demand through its read-only console tools. The notebook itself does **not** stitch files together.

</details>

## 3. Inspect the artifact (no agent attached yet)

<details>
<summary>Details</summary>

Two small helpers let us look at the artifact before we attach an agent:

- `get_artifact_tree(path)` → stringified directory tree (also reused to inject into the system prompt later).
- `inspect_artifact(path)` → prints it.

</details>

In [ ]:
def get_artifact_tree(artifact_path: str | Path) -> str:
    """Return a stringified directory tree of the artifact (markdown-friendly)."""
    root = Path(artifact_path).resolve()
    lines: list[str] = [f"{root.name}/"]

    def walk(d: Path, prefix: str = "") -> None:
        entries = sorted(d.iterdir(), key=lambda p: (p.is_file(), p.name))
        for i, p in enumerate(entries):
            connector = "└── " if i == len(entries) - 1 else "├── "
            lines.append(f"{prefix}{connector}{p.name}{'/' if p.is_dir() else ''}")
            if p.is_dir():
                extension = "    " if i == len(entries) - 1 else "│   "
                walk(p, prefix + extension)

    walk(root)
    return "\n".join(lines)


def inspect_artifact(artifact_path: str | Path) -> None:
    """Print the artifact tree for human inspection before attaching an agent."""
    print(get_artifact_tree(artifact_path))

In [ ]:
inspect_artifact(ARTIFACT_PATH)

## 4. Assembling the agent from the artifact

<details>
<summary>Details</summary>

`create_agent_from_artifact` builds the system prompt from three pieces:

1. The verbatim contents of `agents.md` — already a fully written agent prompt.
2. The stringified artifact tree — so the agent knows the layout without `ls`-ing from scratch.
3. A short **navigation directive** — telling the agent to read each subdirectory's `index.md` first, prefer targeted `read_file` calls over `grep`, and respect the read-only filesystem.

It then wires up `LocalBackend` (rooted at the artifact path, `enable_execute=False`, `READONLY_RULESET` permissions) and a matching `ConsoleCapability`. A `Deps` dataclass exposes the backend to the console toolset.

A placeholder `echo` tool is registered as a stand-in for the real Prithvi inference tools (`geocode_location`, `check_hls_availability`, `run_prithvi_inference`, …). Those land in a follow-up.

When `debug=True`, the finalized system prompt is rendered inline as markdown (via `IPython.display.Markdown`) so you can see exactly what the model receives.

</details>

In [ ]:
NAVIGATION_DIRECTIVE = """
# Artifact navigation

You are running against a local artifact workspace, mounted read-only at the root
of your console tools (`ls`, `read_file`, `glob`, `grep`). The full layout is:

```
{tree}
```

Navigation rules:
- When you need details about a subdirectory (e.g. `contexts/`, `tools/`, `guardrails/`),
  always `read_file` its `index.md` FIRST. The index manifests describe what each
  file in that directory is for. Then read only the specific files you need.
- Prefer `read_file` for targeted reads. **Do NOT use `grep`** — the tree above
  and each subdirectory's `index.md` already tell you exactly what's available
  and what it contains, so keyword searching is unnecessary and wastes a turn.
  Similarly, avoid blanket `ls` of directories you've already seen in the tree.
- The filesystem is read-only. Do not attempt to write, edit, or execute.
""".strip()


@dataclass
class Deps:
    backend: LocalBackend


def create_agent_from_artifact(
    artifact_path: str | Path = ARTIFACT_PATH,
    model: str = MODEL,
    model_settings: dict | None = None,
    debug: bool = False,
) -> tuple[Agent, Deps]:
    artifact_path = Path(artifact_path).resolve()

    # Assemble system prompt: agents.md + tree + navigation directive.
    agents_md = (artifact_path / "agents.md").read_text()
    tree = get_artifact_tree(artifact_path)
    system_prompt = (
        f"{agents_md}\n\n"
        f"{NAVIGATION_DIRECTIVE.format(tree=tree)}"
    )

    backend = LocalBackend(
        root_dir=artifact_path,
        allowed_directories=[str(artifact_path)],
        enable_execute=False,
        permissions=READONLY_RULESET,
    )
    capability = ConsoleCapability(
        include_execute=False,
        permissions=READONLY_RULESET,
    )

    agent = Agent(
        model=model,
        capabilities=[capability],
        system_prompt=system_prompt,
        deps_type=Deps,
        model_settings=model_settings or MODEL_SETTINGS,
    )

    # Placeholder tool — to be replaced with real Prithvi-side tools later.
    @agent.tool_plain
    def echo(message: str) -> str:
        """Echo back the given message. Placeholder for real inference tools."""
        return f"echo: {message}"

    if debug:
        display(Markdown(
            f"**Finalized system prompt** — {len(system_prompt)} chars\n\n"
            f"---\n\n"
            f"{system_prompt}"
        ))

    return agent, Deps(backend=backend)

## 5. Initialize the agent

In [ ]:
agent, deps = create_agent_from_artifact(debug=True)

## 6. Run (async)

<details>
<summary>Details</summary>

`Agent.run(...)` is async and returns the full result once the agent is done. Jupyter lets us `await` it directly inside a cell.

</details>

In [ ]:
result = await agent.run(
    "List the tools available in this artifact and briefly summarize what each one does.",
    deps=deps,
)
print(result.output)

## 7. Run with streaming

<details>
<summary>Details</summary>

For longer answers it's nicer to stream tokens as they arrive instead of waiting for the full reply. `agent.run_stream(...)` yields the output incrementally so we can print each chunk as it comes in.

</details>

In [ ]:
async with agent.run_stream(
    "List the tools available in this artifact and briefly summarize what each one does.",
    deps=deps,
) as result:
    async for chunk in result.stream_text(delta=True):
        print(chunk, end="", flush=True)
    print()

## 8. What's next

<details>
<summary>Details</summary>

- Replace the `echo` dummy with real inference tools described in `tools/index.md` (`geocode_location`, `check_hls_availability`, `run_prithvi_inference`, `get_prithvi_job_status`, `get_prithvi_results`, plus the auxiliary dataset signal tools).
- Try out-of-scope queries to confirm the agent refuses per the `guardrails/` rules.
- Swap `ARTIFACT_PATH` to a different artifact directory — the same `create_agent_from_artifact` call gives you a different agent without code changes.

</details>